# Global CESM+WW3 hindcast setup (Jan 2000)

See `notes.md` in this directory for the full research/decisions writeup. Summary:

- Compset `GW_JRA` (MOM6+CICE+WW3, JRA55-do v1.5 winds), grid `TL319_t232_wg37`, checkout `~/work/installs/cesm3_0_beta08_gui`.
- Run `1999-12-01` → spin-up buffer, through end of Jan 2000.
- Point/spectral output (`points.list`) is **not** staged automatically by WW3's `buildnml` — confirmed by reading it directly. This notebook builds it and copies it into `$RUNDIR` by hand between `case.setup` and `case.submit`.
- This session runs on a Casper node; CIME's `case.setup`/`case.build`/`case.submit`/`xmlchange` all enforce a machine-match check against `derecho`. `ssh derecho` works from here (confirmed 2026-08-12) — every CIME lifecycle command below is run that way for consistency. Plain file edits (user_nl_*, points.list) don't need ssh since GLADE is shared.
- **`case.submit` at the end is NOT run automatically — confirm before running that cell.** It consumes real queue time/allocation on a ~62-day global run.

In [ ]:
import subprocess
from pathlib import Path

CESMROOT = Path("~/work/installs/cesm3_0_beta08_gui").expanduser()
CASE_NAME = "g.ww3hindcast.jra.TL319_t232_wg37.2000-01"
CASEROOT = Path("~/croc_cases").expanduser() / CASE_NAME
PROJECT = "ncgd0011"
MACHINE = "derecho"
COMPSET = "GW_JRA"
RES = "TL319_t232_wg37"

RUN_STARTDATE = "1999-12-01"
# 63 days: 31 (Dec 1999) + 31 (Jan 2000) + 1, so the run's last full day is
# 2000-01-31 (a run of exactly 62 days would stop AT 2000-01-31 00Z, i.e.
# before that day's output is written). Adjust here if the spin-up length
# changes.
STOP_OPTION = "ndays"  # NOT "nday" -- valid values are none/never/nsteps/nseconds/
                        # nminutes/nhours/ndays/nmonths/nyears/date/end (confirmed
                        # via a real xmlchange ERROR message during the 2026-08-12 shakeout)
STOP_N = 63

POINTS_LIST_STRIDE = 2  # see generate_points_list.py; raise to cut point-output volume


def run_on_derecho(cmd, cwd=None):
    """Run a CIME lifecycle command on an actual derecho login node via ssh
    (this session is on Casper -- CIME enforces a machine-match check that
    fails for case.setup/case.build/case.submit/xmlchange run from here
    directly; see notes.md)."""
    full_cmd = f"cd {cwd} && {cmd}" if cwd else cmd
    result = subprocess.run(
        ["ssh", "derecho", full_cmd], capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"Command failed on derecho (rc={result.returncode}): {cmd}")
    return result

## Step 1 — create_newcase

In [ ]:
create_newcase_cmd = (
    f"{CESMROOT}/cime/scripts/create_newcase "
    f"--case {CASEROOT} --compset {COMPSET} --res {RES} "
    f"--machine {MACHINE} --project {PROJECT} --run-unsupported"
)
run_on_derecho(create_newcase_cmd)

## Step 2 — xmlchange (dates, PE layout left at default for a first shakeout run)

Per notes.md: no walltime/PE-layout estimate exists yet for a real 62-day global run at this resolution — the only existing `GW_JRA`/`TL319_t232_wg37` tests are 1-day smoke tests. Leaving `NTASKS`/`JOB_WALLCLOCKTIME` at the compset default for now; revisit after a short shakeout submission (see Step 6 comment).

In [ ]:
xmlchanges = [
    f"RUN_STARTDATE={RUN_STARTDATE}",
    f"STOP_OPTION={STOP_OPTION}",
    f"STOP_N={STOP_N}",
]
for change in xmlchanges:
    run_on_derecho(f"./xmlchange {change}", cwd=CASEROOT)

## Step 3a — SourceMods fix: point output is silently disabled without this

**Important finding while writing this notebook** (not assumed going in): `namelist_definition_ww3.xml` exposes `type%point%file` (turns point output on) but never defines `date%point%start/stride/stop` — confirmed by grepping every "point"-related entry in the file, only one hit. The underlying Fortran (`model/src/w3nmlshelmd.F90`, `READ_OUTPUT_DATE_NML`) defaults `DATE%POINT%STRIDE` to `'0'` when absent — WW3's own convention for "off". So without a fix, point output stays disabled even with `points.list` staged correctly.

`buildnml` explicitly supports overriding the whole namelist definition via `$CASEROOT/SourceMods/src.ww3/namelist_definition_ww3.xml` (its own comment: "user definition *replaces* existing definition") — `add_point_output_dates.py` in this directory copies the real file and adds the three missing entries there, rather than patching the checked-out component in place. Must run after `create_newcase` (needs `SourceMods/src.ww3/` to already exist) and before `case.build` (so `buildnml` picks up the override).

In [ ]:
import subprocess as sp

sp.run(
    ["python3", "add_point_output_dates.py", str(CASEROOT)],
    check=True,
)

## Step 3b — user_nl_ww3: enable point/spectral output

Plain file edit — GLADE is shared, no ssh needed. `type_percent_point_percent_file` maps to WW3's `type%point%file` namelist var (see `components/ww3/cime_config/user_nl_ww3`'s own header comment for the `_percent_` → `%` translation rule). Hourly cadence matches `ERA5_WAVE_SPECTRA`'s cadence in CrocoDash's existing WW3 forcing pipeline. Only valid now that Step 3a's SourceMods override exists.

In [ ]:
user_nl_ww3_additions = """
! --- point/spectral output for CESM_WW3_OUTPUT product (added by setup_case.ipynb) ---
type_percent_point_percent_file = 'points.list'
date_percent_point_percent_start = '{start} 0'
date_percent_point_percent_stride = '3600'
date_percent_point_percent_stop = '99990101 0'
""".format(start=RUN_STARTDATE.replace("-", ""))
# Values must be quoted as Fortran character literals -- confirmed the hard
# way during the 2026-08-12 shakeout: CIME's NamelistDefinition.is_valid_value
# calls is_valid_fortran_namelist_literal(type_, scalar) on each whitespace-
# tokenized piece of an unquoted value, and since these entries are declared
# type=char, an unquoted "19991201 0" gets split into two tokens and rejected
# ("Invalid values ['19991201', '0']"), and even a single bare numeric token
# like "3600" is rejected as not-a-valid-character-literal. Wrapping every
# value in single quotes (matching type_percent_point_percent_file's own
# style) fixed both.

user_nl_ww3_path = CASEROOT / "user_nl_ww3"
with open(user_nl_ww3_path, "a") as f:
    f.write(user_nl_ww3_additions)

print(user_nl_ww3_path.read_text())

## Step 4 — case.setup, then stage points.list into $RUNDIR

`case.setup` creates `$RUNDIR` but WW3's `buildnml` never copies a `points.list` there (confirmed by reading `components/ww3/cime_config/buildnml`) — it has to be placed by hand before the model actually runs.

In [ ]:
run_on_derecho("./case.setup", cwd=CASEROOT)

In [ ]:
import subprocess as sp

# Absolute path (not relative to CASEROOT) since generate_points_list.py
# writes to this notebook's own working directory -- assumed to be this dev
# directory. GLADE is shared, so an absolute /glade/... path is reachable
# identically whether read from Casper or, below, from derecho over ssh.
points_list_path = Path.cwd() / "points.list"
sp.run(
    [
        "python3",
        "generate_points_list.py",
        "--stride", str(POINTS_LIST_STRIDE),
        "--out", str(points_list_path),
    ],
    check=True,
)

# RUNDIR: use ./xmlquery over ssh, not a manual env_run.xml parse -- RUNDIR is
# stored there as an unresolved `value="$CIME_OUTPUT_ROOT/$CASE/run"` XML
# attribute (confirmed 2026-08-12), not plain resolved text. xmlquery itself
# is safe to run here because we're invoking it FROM an actual derecho node
# via ssh -- the machine-match restriction only bites when it's run directly
# from this (Casper) session.
result = run_on_derecho("./xmlquery RUNDIR", cwd=CASEROOT)
rundir = result.stdout.split("RUNDIR:")[1].strip()
print("RUNDIR:", rundir)

run_on_derecho(f"mkdir -p {rundir} && cp {points_list_path} {rundir}/")

## Step 5 — case.build

In [ ]:
run_on_derecho("./case.build", cwd=CASEROOT)

## Step 6 — case.submit (DO NOT run without confirming first)

This is the resource-consuming step: real queue time and `ncgd0011` allocation
on a ~62-day global run. Recommend a short shakeout first (e.g. temporarily
set `STOP_N` to a few days via `xmlchange`, submit, check walltime/cost, then
reset `STOP_N` back to the full window) before committing to the full run.

In [ ]:
# Uncomment when ready:
# run_on_derecho("./case.submit", cwd=CASEROOT)